In [8]:
import torch
import cv2
import numpy as np
from torchvision.utils import save_image
import os
os.makedirs("outputs", exist_ok=True)


class AutoEncoderAndDecoder(torch.nn.Module):
    def __init__(self):
        super(AutoEncoderAndDecoder,self).__init__()
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(100*100*3,5000), 
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout(0.0, inplace=True),
            
            torch.nn.Linear(5000, 2500),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(2500, 1000),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(1000, 500),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(500, 100),
        )
        self.decoder = torch.nn.Sequential(
            torch.nn.Linear(100, 500),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(500, 1000),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(1000, 2500),
            torch.nn.ReLU(inplace=True),           
            torch.nn.Linear(2500, 5000),
            torch.nn.ReLU(inplace=True),           
            torch.nn.Linear(5000, 100*100*3),
        )
            
    def forward(self, x):
        y_enc = self.encoder(x)
        y_dec = self.decoder(y_enc)
        return y_enc, y_dec

model = AutoEncoderAndDecoder()

# 数据处理
img = cv2.imread("p1.jpg")
img = cv2.resize(img,(100,100))
img = img.astype(np.float32) / 255.0 

x = torch.tensor(img, dtype=torch.float32)
x = x.permute(2,0,1)
x = torch.unsqueeze(x, dim=0) 

# 优化器
opt = torch.optim.Adam(model.parameters(), lr=0.001)
floss = torch.nn.MSELoss(reduction="mean")

# 训练
epoches = 10
for e in range(epoches):
    v_loss = 0.0
    for i in range(100):
        opt.zero_grad()
        
        x_in = x.reshape(-1, 100*100*3)     # 形状：1×30000 匹配模型输入
        _, y = model(x_in)
        
        loss = floss(y, x_in)
        loss.backward()
        opt.step()
        v_loss += loss.item()
        
    print(f"epoch {e} loss: {v_loss/100:.4f}")

    save_image(y.view(-1,3,100,100), f"outputs/img_{e}.jpg")
    torch.save(model.state_dict(), "img.pt")

epoch 0 loss: 4.8001
epoch 1 loss: 0.0000
epoch 2 loss: 0.0000
epoch 3 loss: 0.0000
epoch 4 loss: 0.0000
epoch 5 loss: 0.0000
epoch 6 loss: 0.0000
epoch 7 loss: 0.0000
epoch 8 loss: 0.0000


KeyboardInterrupt: 